# What does this Script do?

## Imports the Relevant data from the Partners in care Looker database, cleans and combines it, then updates a SQl table (partners_in_care_veterans_summary) where it's used to maintain a PowerBI Dashboard.  

import python packages

In [22]:
import pandas as pd
import os
import json
import datetime
from datetime import datetime, date, timezone
import numpy as np
import pytz


from io import StringIO
import looker_sdk

import custom packages

In [23]:
from utils.ds_general_utils import DsEmail
from utils.ds_general_utils import ConfigTools
from utils.ds_general_utils import Task, TaskManager
from utils.ds_general_utils import MultiThreadedLogger


from utils.sql_connection import PicHorneDashesConnection

initialize task tracker

The task tracker allows for watching individual cell executions and seeing which ones fail, as well as preventing future cells from running if an error occurs. it also generates a summary email upon failure. 

In [24]:
# initialization of the task manager with the overall name of the task this code is supposed to do.
task_mgr = TaskManager("partners in care clarity looker veterans data ingestion")

initialize logger

This cell initializes the logger, which is used to log information, warnings, and errors during the execution of the script. It creates a log file with a timestamp in the filename and sets up the logger with a specific format and logging level. If the logging directory doesn't exist it creates it and moves all but the last 5 logs to the archive

In [25]:
task = Task(task_nm="Initialize log file", task_details="")
try:
    # creating any relevant dates
    now = datetime.now()
    now_str = now.strftime("%Y%m%d_%H%M%S")
    # log file info
    logging_dir = os.path.join(os.getcwd(), "logs")
    logging_filename = (
        "partners_in_care_clarity_looker_veterans_data_ingestion_{}.log".format(now_str)
    )
    logging_filename_with_path = os.path.join(logging_dir, logging_filename)

    logger_manager = MultiThreadedLogger(
        logging_filename_with_path,
        "Partners_In_Care",
        "Veterans_Data_ingestion",
        level="INFO",
        format_string="%(asctime)s %(levelname)s %(message)s",
    )
    logger = logger_manager.logger
    # updating the task tracker
    task["Details"] = "Successfully created and began writing to log file"
    task["Success"] = True
    logger.info("Data ingestion for partners in care veterans started")

except Exception as e:
    task["Details"] = str(e)
    print(e)

task_mgr.add_task(task)

initialize variables

The below cell initializes starting global variables such as directory paths, configuration objects, and various date and time variables. It also updates the task tracker with the status of the initialization process.

In [26]:
# defined outside the try so the failure-alert cell at the end always has them
today = date.today()
today_str = today.strftime("%m_%d_%Y")
reporting_period_today_str = today.strftime("%Y-%m-%d")

task = Task(task_nm="Initialize starting global variables", task_details="")
try:
    task_mgr.check_errors()
    main_directory_path = str(os.getcwd())
    config_manager = ConfigTools(main_directory_path)
    config_object = config_manager.config_data
    looker_config_path = os.path.join(main_directory_path, "utils", "looker.ini")

    # variables used in pulling records from looker
    model = "oahu_connection_model"
    view = "client"
    view_base = "base"
    limit = 100000

    now_utc = datetime.now(timezone.utc)
    hst = pytz.timezone("Pacific/Honolulu")
    now_hst = datetime.now(hst)
    now_utc_str = now_utc.strftime("%Y/%m/%d %H:%M:%S")
    now_str_formatted = now.strftime("%Y/%m/%d %H:%M:%S")
    now_hst_str = now_hst.strftime("%Y/%m/%d %H:%M:%S")

    # updating the task tracker
    task["Details"] = "successfully initialize global variables"
    task["Success"] = True
    logger.info("Initialized starting global variables")
except Exception as e:
    task["Details"] = str(e)
    logger.error(f"Initializing global variables failed with the following error :{e}")
    print(e)

task_mgr.add_task(task)

Grab program level information 

The below cell connects to the database and ingests veteran data from Looker. It initializes the Looker SDK, defines the queries to be executed, runs the queries, processes the results, and merges the data into a single DataFrame. All queries have a common filter of client veteran status of yes and head of household of yes. The task tracker is updated with the status of the data ingestion process.

In [27]:
task = Task(
    task_nm="Connect to database and ingest veteran data from looker",
    task_details="",
)
try:
    task_mgr.check_errors()
    # initialize sdk object to pull in data from looker
    sdk = looker_sdk.init40(looker_config_path)
    # create a list of queries to pull in data from looker
    queries = [
        {
            "fields": [
                "client.id",
                "client.full_name",
                "client.unique_identifier",
                "programs.name",
                "programs.project_type_code",
                "enrollments.id",
                "enrollments.start_date",
                "enrollments.end_date",
                "client_referral_status.Currently_On_Queue",
                "chronically_homeless_at_entry.is_chronic_homeless",
                "entry_screen.disabling_condition",
                "entry_screen.chronic_7_date",
            ],
            "filters": {
                "static_demographics.veteran_text": "yes",
                "entry_screen.head_of_household": "yes",
            },
            "name": "clients_and_programs_df",
        },
        {
            "fields": [
                "clients.id",
                "programs.project_type_code",
                "enrollments.start_date",
                "enrollments.end_date",
            ],
            "filters": {
                "static_demographics.veteran_text": "yes",
                "entry_screen.head_of_household": "yes",
                "enrollments.date_filter": "",
                "programs.name": "-GPD Aftercare - Retention Housing CM,-GPD Aftercare - Retention Housing Collaborative CM",
                "entry_screen.move_in_date": "null",
            },
            "name": "count_base_df",
        },
        {
            "fields": [
                "client.id",
                "screens.name",
                "client_assessments.assessment_date",
            ],
            "filters": {
                "static_demographics.veteran_text": "yes",
                "entry_screen.head_of_household": "yes",
                "screens.name": "Custom VI-SPDAT Families (V2), Custom VI-SPDAT Individual Adult (V2), Custom VI-SPDAT Transition Age Youth (V1)",
            },
            "name": "assessments_df",
        },
        {
            "fields": [
                "client.id",
                "client_assessments.assessment_date",
                "client_assessment_custom.c_discharge_custom",
            ],
            "filters": {
                "static_demographics.veteran_text": "yes",
                "entry_screen.head_of_household": "yes",
                "client_assessment_custom.c_discharge_custom": "-null",
            },
            "name": "discharge_df",
        },
        {
            "fields": [
                "client.id",
                "client_file_names.name",
            ],
            "filters": {
                "static_demographics.veteran_text": "yes",
                "entry_screen.head_of_household": "yes",
                "client_file_names.name": "DD 214 (Veterans Only)",
            },
            "name": "files_df",
        },
        {
            "fields": [
                "client.id",
                "referral_connection_enrollments.id",
                "referral_connection_enrollment_programs.name",
                "referrals.status",
                "referrals.id",
                "referrals.date_accepted_in_program_date",
                "referrals.created_date",
                "referrals.added_date",
                "referto_program.project_type_code",
                "referto_program.name",
            ],
            "filters": {
                "static_demographics.veteran_text": "Yes",
                "entry_screen.head_of_household": "Yes",
                "referto_program.name": "-null",
            },
            "name": "referrals_df",
        },
    ]

    # Function to generate query body
    def generate_body(model, view, query, limit):
        return {
            "model": model,
            "view": view,
            "fields": query["fields"],
            "filters": query["filters"],
            "limit": limit,
        }

    # Function to run the query
    def run_query(sdk, model, view, query, limit):
        body = generate_body(model, view, query, limit)
        result = sdk.run_inline_query(result_format="csv", body=body)
        return result

    # Function to process the results
    def process_result(result):
        df = pd.read_csv(StringIO(result))
        return df

    # begin execution of the queries

    veterans_df = None
    # Loop through the queries and run them
    for query in queries:
        if query["name"] == "count_base_df":
            retries = 3  # Number of retries
            for attempt in range(retries):  # Retry up to the specified number of times
                try:
                    result = run_query(
                        sdk, model, view_base, query, limit
                    )  # Run the query
                    break  # Exit loop if successful
                except Exception as e:
                    if (
                        attempt < retries - 1
                    ):  # Log and retry for all but the last attempt
                        logger.warning(
                            f"Attempt {attempt + 1} to fetch data for program '{query['name']}' failed: {e}. Retrying..."
                        )
                        continue
                    else:  # Raise the exception after all retries fail
                        logger.error(
                            f"All {retries} attempts to fetch data for program '{query['name']}' failed: {e}"
                        )
                        raise
            base_vets_df = process_result(result)
            continue
        result = run_query(sdk, model, view, query, limit)  # Run the query
        df = process_result(result)

        # Process the data for specific queries that need it

        if (
            query["name"] == "discharge_df"
        ):  # Process the discharge data to keep only the latest discharge status based on assessment date
            df["Client Assessments Assessment Date"] = pd.to_datetime(
                df["Client Assessments Assessment Date"]
            )

            # Sort the dataframe by assessment date in descending order
            df_sorted = df.sort_values(
                by="Client Assessments Assessment Date", ascending=False
            )

            # Drop duplicates based on client ID, keeping only the latest entry
            df_latest = df_sorted.drop_duplicates(
                subset="Clients Client ID", keep="first"
            )

            # Drop the assessment date column
            df = df_latest.drop(columns=["Client Assessments Assessment Date"])

        if veterans_df is None:
            veterans_df = df
        else:
            if (
                query["name"] == "referrals_df"
            ):  # separating out referrals into completed and pending referrals and merging them with the veterans data separately

                df["Referral Connected Enrollments Enrollment ID"] = (
                    df["Referral Connected Enrollments Enrollment ID"]
                    .fillna(-1)
                    .astype(int)
                )  # Convert to int to match the veterans data and fill NA with -1 as this will never be a valid enrollment ID

                pending_statuses = [
                    "Pending",
                    "Pending - In Process",
                    "Denied",
                    "Expired",
                ]  # Define the statuses that are considered pending

                completed_referrals = df.loc[
                    df["Referrals Status"] == "Completed",
                    [
                        "Referral Connected Enrollments Enrollment ID",
                        "Referrals ID",
                        "Referrals Created Date",
                    ],
                ]  # Filter out the completed referrals

                pending_referrals = df.loc[
                    df["Referrals Status"].isin(pending_statuses),
                    ["Clients Client ID", "Referrals ID", "Referrals Created Date"],
                ]  # Filter out the pending referrals

                veterans_df = pd.merge(
                    veterans_df,
                    completed_referrals,
                    how="left",
                    left_on="Enrollments Enrollment ID",
                    right_on="Referral Connected Enrollments Enrollment ID",
                )  # Merge on the enrollment ID for completed referrals

                veterans_df = pd.merge(
                    veterans_df,
                    pending_referrals,
                    how="left",
                    on="Clients Client ID",
                    suffixes=("_completed", "_pending"),
                )  # Merge on the client ID for pending referrals

            else:
                veterans_df = pd.merge(
                    veterans_df, df, on="Clients Client ID", how="left"
                )  # Merge the dataframes

    veterans_df.sort_values(
        by="Clients Client ID", inplace=True
    )  # Sort the dataframe by client ID

    # updating the task tracker
    task["Details"] = "Successfully grabbed veterans data from looker"
    task["Success"] = True
    logger.info("Connected to database and pulled veterans data from looker")

except Exception as e:
    task["Details"] = str(e)
    logger.error(
        f"connecting to database and grabbing veterans data failed with the following error :{e}"
    )
    print(e)

task_mgr.add_task(task)

Generate aggregated table of veterans summary

The below cell prepares for the aggregation of counting of veteran categories by converting any date columns to an actual datetime object allowing for easier comparison of dates later. 

In [28]:
task = Task(
    task_nm="Convert date columns to datetime for comparison later",
    task_details="",
)
try:
    task_mgr.check_errors()
    veterans_df["Enrollments Project Start Date"] = pd.to_datetime(
        veterans_df["Enrollments Project Start Date"], errors="coerce"
    ).copy()

    veterans_df["Enrollments Project Exit Date"] = pd.to_datetime(
        veterans_df["Enrollments Project Exit Date"], errors="coerce"
    ).copy()

    veterans_df["Client Assessments Assessment Date"] = pd.to_datetime(
        veterans_df["Client Assessments Assessment Date"], errors="coerce"
    ).copy()

    veterans_df["Referrals Created Date_completed"] = pd.to_datetime(
        veterans_df["Referrals Created Date_completed"], errors="coerce"
    ).copy()

    veterans_df["Referrals Created Date_pending"] = pd.to_datetime(
        veterans_df["Referrals Created Date_pending"], errors="coerce"
    ).copy()

    veterans_df[
        "Entry Screen Approximate Date this Episode of Homelessness Started Date"
    ] = pd.to_datetime(
        veterans_df[
            "Entry Screen Approximate Date this Episode of Homelessness Started Date"
        ],
        errors="coerce",
    ).copy()

    veterans_df["Referrals Created Date_completed"] = pd.to_datetime(
        veterans_df["Referrals Created Date_completed"], errors="coerce"
    ).copy()

    veterans_df["Referrals Created Date_pending"] = pd.to_datetime(
        veterans_df["Referrals Created Date_pending"], errors="coerce"
    ).copy()

    # updating the task tracker
    task["Details"] = "Successfully converted columns to datetime"
    task["Success"] = True
    logger.info("Converted columns to datetime")

except Exception as e:
    task["Details"] = str(e)
    logger.error(f"converting columns to datetime failed with the following error :{e}")
    print(e)

task_mgr.add_task(task)

Start of actually calculating values for summary. Row numbers correspond to Monthly Vet BNL Data spreadsheet. All calculations are done for the past 12 whole months. (if today is October 15, 2023, the past 12 whole months would be from October 1, 2022, to September 30, 2023.)

Count of veterans in HMIS (row 2)

This cell calculates the total self-identified veterans in HMIS using the base model. all other calculations will use the CE model. It iterates over the past 12 months, generating the last date of each month and filtering the `base_vets_df` DataFrame to include only those veterans who were enrolled in a non-Homelessness Prevention type code program as of the last day of the month. The count of unique client IDs is then stored in the `result_veterans_df` DataFrame under the column "Self Identified Vets in HMIS". The task tracker is updated with the status of the calculation process. (Note if you need to change this to count on CE again just change base_vets_df to veterans_df)

In [29]:
task = Task(
    task_nm="Calculate count of total self identified veterans in HMIS",
    task_details="",
)
try:
    task_mgr.check_errors()

    base_vets_df["Enrollments Project Start Date"] = pd.to_datetime(
        base_vets_df["Enrollments Project Start Date"], errors="coerce"
    ).copy()

    base_vets_df["Enrollments Project Exit Date"] = pd.to_datetime(
        base_vets_df["Enrollments Project Exit Date"], errors="coerce"
    ).copy()

    # Create a new DataFrame to store the results

    result_veterans_df = pd.DataFrame(columns=["Month", "Self Identified Vets in HMIS"])

    # Generate the last date of each month for the past 12 months

    for i in range(1, 13):

        last_date_of_month = (now - pd.DateOffset(months=i)).replace(
            day=1
        ) + pd.DateOffset(months=1, days=-1)

        self_identified_vets = base_vets_df[
            (base_vets_df["Enrollments Project Start Date"] <= last_date_of_month)
            & (
                (base_vets_df["Enrollments Project Exit Date"] > last_date_of_month)
                | (base_vets_df["Enrollments Project Exit Date"].isna())
            )
            & (base_vets_df["Programs Project Type Code"] != "Homelessness Prevention")
        ].copy()  # Filter the data to only include clients who were enrolled in an eligible program at the end of the month

        count = self_identified_vets[
            "Clients Client ID"
        ].nunique()  # Count the number of unique clients

        result_veterans_df = pd.concat(
            [
                result_veterans_df,
                pd.DataFrame(
                    {
                        "Month": [last_date_of_month.strftime("%Y-%m")],
                        "Self Identified Vets in HMIS": [count],
                    }
                ),
            ],
            ignore_index=True,
        )  # Append the results to the DataFrame

    result_veterans_df.set_index("Month", inplace=True)  # Set the index to the month

    # updating the task tracker
    task["Details"] = (
        "Successfully calculated count of total self identified veterans in HMIS"
    )
    task["Success"] = True
    logger.info("Calculated count of total self identified veterans in HMIS")

except Exception as e:
    task["Details"] = str(e)
    logger.error(
        f"calculating count of total self identified veterans in HMIS failed with the following error :{e}"
    )
    print(e)

task_mgr.add_task(task)

Count of Coordinated Entry Vets (Row 3)

This cell calculates the count of self-identified veterans in HMIS who are enrolled in the program coordinated entry program. It iterates over the past 12 months, generating the last date of each month and filtering the `veterans_df` DataFrame to include only those veterans who were enrolled in a coordinated entry program as of the last day of the month. The count of unique client IDs is then stored in the `result_veterans_df` DataFrame under the column "Coordinated Entry Vets in HMIS". The task tracker is updated with the status of the calculation process.

In [30]:
task = Task(
    task_nm="Calculate count of self identified veterans in HMIS in coordinated entry programs",
    task_details="",
)
try:
    task_mgr.check_errors()  # Create a new DataFrame to store the results

    result_veterans_df["Coordinated Entry Vets in HMIS"] = 0

    # Generate the last date of each month for the past 12 months

    for i in range(1, 13):

        last_date_of_month = (now - pd.DateOffset(months=i)).replace(
            day=1
        ) + pd.DateOffset(months=1, days=-1)

        coordinated_entry_vets = veterans_df[
            (veterans_df["Enrollments Project Start Date"] <= last_date_of_month)
            & (
                (veterans_df["Enrollments Project Exit Date"] > last_date_of_month)
                | (veterans_df["Enrollments Project Exit Date"].isna())
            )
            & (veterans_df["Programs Name"] == "Coordinated Entry Program")
        ].copy()  # Filter the data to only include clients who were enrolled in the coordinated entry program at the end of the month

        count_coordinated_entry = coordinated_entry_vets[
            "Clients Client ID"
        ].nunique()  # Count the number of unique clients

        result_veterans_df.loc[
            last_date_of_month.strftime("%Y-%m"), "Coordinated Entry Vets in HMIS"
        ] = count_coordinated_entry  # Append the results to the DataFrame

    # updating the task tracker
    task["Details"] = (
        "Successfully calculated count of self identified veterans in HMIS in coordinated entry programs"
    )
    task["Success"] = True
    logger.info(
        "Calculated count of self identified veterans in HMIS in coordinated entry programs"
    )

except Exception as e:
    task["Details"] = str(e)
    logger.error(
        f"calculating count of self identified veterans in HMIS in coordinated entry programs failed with the following error :{e}"
    )
    print(e)

task_mgr.add_task(task)

Count of vets on BNL (row 5)

This cell calculates the count of self-identified veterans who are on the By-Name List, which is those from the previous filter plus having had a VI-SPDAT in the past 365 days. It iterates over the past 12 months, generating the last date of each month and filtering the `veterans_df` DataFrame to include only those veterans who were enrolled in a coordinated entry program as of the last day of the month and those with a VI-SPDAT in the past 365 days from the last date of the month. The count of unique client IDs is then stored in the `result_veterans_df` DataFrame under the column "Vets on BNL". The task tracker is updated with the status of the calculation process.

In [31]:
task = Task(
    task_nm="Calculate count of veterans on BNL",
    task_details="",
)
try:
    task_mgr.check_errors()  # Create a new column to store the results

    result_veterans_df["Vets on BNL"] = 0

    # Generate the last date of each month for the past 12 months

    for i in range(1, 13):

        last_date_of_month = (now - pd.DateOffset(months=i)).replace(
            day=1
        ) + pd.DateOffset(months=1, days=-1)

        one_year_ago = last_date_of_month - pd.DateOffset(
            years=1
        )  # Calculate the date one year ago

        bnl_vets = veterans_df[
            (veterans_df["Enrollments Project Start Date"] <= last_date_of_month)
            & (
                (veterans_df["Enrollments Project Exit Date"] > last_date_of_month)
                | (veterans_df["Enrollments Project Exit Date"].isna())
            )
            & (veterans_df["Programs Name"] == "Coordinated Entry Program")
            & (veterans_df["Client Assessments Assessment Date"] >= one_year_ago)
            & (veterans_df["Client Assessments Assessment Date"] <= last_date_of_month)
        ].copy()  # Filter the data to only include clients who were enrolled in the coordinated entry program at the end of the month and had an assessment in the past year

        count_vets_on_bnl = bnl_vets[
            "Clients Client ID"
        ].nunique()  # Count the number of unique clients

        result_veterans_df.loc[last_date_of_month.strftime("%Y-%m"), "Vets on BNL"] = (
            count_vets_on_bnl  # Append the results to the DataFrame
        )

    # updating the task tracker
    task["Details"] = "Successfully calculated count of veterans on BNL"
    task["Success"] = True
    logger.info("Calculated count of veterans on BNL")

except Exception as e:
    task["Details"] = str(e)
    logger.error(
        f"calculating count of veterans on BNL failed with the following error :{e}"
    )
    print(e)

task_mgr.add_task(task)

Count of vets on BNL with eligible discharge (Row 8)

This cell calculates the count of self-identified veterans who are on the By-Name List with eligible discharge status. It iterates over the past 12 months, generating the last date of each month and filtering the `veterans_df` DataFrame to include only those veterans who were enrolled in a coordinated entry program as of the last day of the month, those with a VI-SPDAT in the past 365 days from the last date of the month, and those with an eligible discharge status. The count of unique client IDs is then stored in the `result_veterans_df` DataFrame under the column "Vets on BNL with Eligible Discharge". The task tracker is updated with the status of the calculation process.

In [32]:
task = Task(
    task_nm="Calculate count of veterans on BNL with eligible discharge",
    task_details="",
)
try:
    task_mgr.check_errors()  # Define the list of eligible discharge statuses

    eligible_discharges = [
        "General",
        "Honorable",
        "Eligible: National Guard/Reserves",
        "Ineligible VHA due to length of active service",
        "Other Than Honorable (OTH)",
        "Bad Conduct Discharge (BCD)",
    ]  # Define the list of eligible discharge statuses

    # Create a new column to store the results

    result_veterans_df["Vets on BNL with Eligible Discharge"] = 0

    # Generate the last date of each month for the past 12 months

    for i in range(1, 13):

        last_date_of_month = (now - pd.DateOffset(months=i)).replace(
            day=1
        ) + pd.DateOffset(months=1, days=-1)

        one_year_ago = last_date_of_month - pd.DateOffset(years=1)

        vets_on_bnl_eligible = veterans_df[
            (veterans_df["Enrollments Project Start Date"] <= last_date_of_month)
            & (
                (veterans_df["Enrollments Project Exit Date"] > last_date_of_month)
                | (veterans_df["Enrollments Project Exit Date"].isna())
            )
            & (veterans_df["Programs Name"] == "Coordinated Entry Program")
            & (veterans_df["Client Assessments Assessment Date"] >= one_year_ago)
            & (veterans_df["Client Assessments Assessment Date"] <= last_date_of_month)
            & (
                veterans_df["Client Assessment Custom Veteran Discharge Status"].isin(
                    eligible_discharges
                )
            )
        ].copy()  # Filter the data to only include clients who were enrolled in the coordinated entry program at the end of the month, had an assessment in the past year, and have an eligible discharge status

        count_vets_on_bnl_eligible = vets_on_bnl_eligible[
            "Clients Client ID"
        ].nunique()  # Count the number of unique clients

        result_veterans_df.loc[
            last_date_of_month.strftime("%Y-%m"), "Vets on BNL with Eligible Discharge"
        ] = count_vets_on_bnl_eligible  # Append the results to the DataFrame

    # updating the task tracker
    task["Details"] = (
        "Successfully calculated count of veterans on BNL with eligible discharge"
    )
    task["Success"] = True
    logger.info("Calculated count of veterans on BNL with eligible discharge")

except Exception as e:
    task["Details"] = str(e)
    logger.error(
        f"calculating count of veterans on BNL with eligible discharge failed with the following error :{e}"
    )
    print(e)

task_mgr.add_task(task)

Count of vets not on community queue (Row 11)

This cell calculates the count of self-identified veterans on the By-Name List with eligible discharge status who are not on the community queue or have an open referral. It iterates over the past 12 months, generating the last date of each month and filtering the `veterans_df` DataFrame to include only those veterans who were enrolled in a coordinated entry program as of the last day of the month, those with a VI-SPDAT in the past 365 days from the last date of the month, those with an eligible discharge status, and those who are not on the community queue or have an open referral. The count of unique client IDs is then stored in the `result_veterans_df` DataFrame under the column "Vets not on Community Queue or Referred". The task tracker is updated with the status of the calculation process.

In [33]:
task = Task(
    task_nm="Calculate count of veterans on BNL with eligible discharge not on community queue or with open referral",
    task_details="",
)
try:
    task_mgr.check_errors()  # Create a new column to store the results

    result_veterans_df["Vets not on Community Queue or Referred"] = 0

    # Generate the last date of each month for the past 12 months

    for i in range(1, 13):

        last_date_of_month = (now - pd.DateOffset(months=i)).replace(
            day=1
        ) + pd.DateOffset(months=1, days=-1)

        one_year_ago = last_date_of_month - pd.DateOffset(years=1)

        vets_with_open_referral_df = veterans_df[
            (
                (veterans_df["Referrals Created Date_completed"] <= last_date_of_month)
                & (veterans_df["Enrollments Project Start Date"] >= last_date_of_month)
            )
            | (veterans_df["Referrals Created Date_pending"] <= last_date_of_month)
        ]  # Filter the data to only include clients who have an open referral at the end of the month

        count_not_on_queue = veterans_df[
            (veterans_df["Enrollments Project Start Date"] <= last_date_of_month)
            & (
                (veterans_df["Enrollments Project Exit Date"] > last_date_of_month)
                | (veterans_df["Enrollments Project Exit Date"].isna())
            )
            & (veterans_df["Programs Name"] == "Coordinated Entry Program")
            & (veterans_df["Client Assessments Assessment Date"] >= one_year_ago)
            & (veterans_df["Client Assessments Assessment Date"] <= last_date_of_month)
            & (
                veterans_df["Client Assessment Custom Veteran Discharge Status"].isin(
                    eligible_discharges
                )
            )
            & (
                veterans_df["Client Referral Status Is client currently on queue?"]
                == "No"
            )
            & (
                ~veterans_df["Clients Client ID"].isin(
                    vets_with_open_referral_df["Clients Client ID"]
                )
            )
        ][
            "Clients Client ID"
        ].nunique()  # Count the number of unique clients who are in the coordinated entry program, had an assessment in the past year,
        # have an eligible discharge status, are not on the community queue, and do not have an open referral

        result_veterans_df.loc[
            last_date_of_month.strftime("%Y-%m"),
            "Vets not on Community Queue or Referred",
        ] = count_not_on_queue  # Append the results to the DataFrame

    # updating the task tracker
    task["Details"] = (
        "Successfully calculated count of veterans on BNL with eligible discharge not on community queue or with open referral"
    )
    task["Success"] = True
    logger.info(
        "Calculated count of veterans on BNL with eligible discharge not on community queue or with open referral"
    )

except Exception as e:
    task["Details"] = str(e)
    logger.error(
        f"calculating count of veterans on BNL with eligible discharge not on community queue or with open referral failed with the following error :{e}"
    )
    print(e)

task_mgr.add_task(task)

count of vets with dd214 (row 15)

This cell calculates the count of self-identified veterans who are on the By-Name List, which is those from the previous filter plus having had a VI-SPDAT in the past 365 days. It iterates over the past 12 months, generating the last date of each month and filtering the `veterans_df` DataFrame to include only those veterans who were enrolled in a coordinated entry program as of the last day of the month and those with a VI-SPDAT in the past 365 days from the last date of the month. The count of unique client IDs is then stored in the `result_veterans_df` DataFrame under the column "Vets on BNL". The task tracker is updated with the status of the calculation process.

In [34]:
task = Task(
    task_nm="Calculate count of veterans with DD214",
    task_details="",
)
try:
    task_mgr.check_errors()

    result_veterans_df["Vets with DD214"] = (
        0  # Create a new column to store the results
    )

    # Generate the last date of each month for the past 12 months

    for i in range(1, 13):

        last_date_of_month = (now - pd.DateOffset(months=i)).replace(
            day=1
        ) + pd.DateOffset(months=1, days=-1)

        one_year_ago = last_date_of_month - pd.DateOffset(
            years=1
        )  # Calculate the date one year ago

        vets_with_dd214 = veterans_df[
            (veterans_df["Enrollments Project Start Date"] <= last_date_of_month)
            & (
                (veterans_df["Enrollments Project Exit Date"] > last_date_of_month)
                | (veterans_df["Enrollments Project Exit Date"].isna())
            )
            & (veterans_df["Programs Name"] == "Coordinated Entry Program")
            & (veterans_df["Client Assessments Assessment Date"] >= one_year_ago)
            & (veterans_df["Client Assessments Assessment Date"] <= last_date_of_month)
            & (
                veterans_df["Client Assessment Custom Veteran Discharge Status"].isin(
                    eligible_discharges
                )
            )
            & (veterans_df["Client Files Client File Name"] == "DD 214 (Veterans Only)")
        ].copy()  # Filter the data to only include clients who were enrolled in the coordinated entry program at the end of the month, had an assessment in the past year, have an eligible discharge status, and have a DD214

        count_with_dd214 = vets_with_dd214[
            "Clients Client ID"
        ].nunique()  # Count the number of unique clients

        result_veterans_df.loc[
            last_date_of_month.strftime("%Y-%m"), "Vets with DD214"
        ] = count_with_dd214  # Append the results to the DataFrame

    # Create a new column to store the percentage of vets with DD214
    result_veterans_df["Vets with DD214 Percentage"] = (
        (
            result_veterans_df["Vets with DD214"]
            / result_veterans_df["Vets on BNL with Eligible Discharge"]
        )
        * 100
    ).round(2)

    # updating the task tracker
    task["Details"] = "Successfully calculated count of veterans with DD214"
    task["Success"] = True
    logger.info("Calculated count of veterans with DD214")

except Exception as e:
    task["Details"] = str(e)
    logger.error(
        f"calculating count of veterans with DD214 failed with the following error :{e}"
    )
    print(e)

task_mgr.add_task(task)

Calculating Chronic vets and sheltered vets (row 23, 33 and 41)

This cell calculates the count of self-identified veterans who are chronically homeless, sheltered, and unsheltered. It iterates over the past 12 months, generating the last date of each month and filtering the `veterans_df` DataFrame to include only those veterans who were enrolled in a coordinated entry program as of the last day of the month, those with a VI-SPDAT in the past 365 days from the last date of the month, and those with an eligible discharge status. It then identifies veterans who are chronically homeless based on specific criteria and counts the number of chronic veterans, sheltered veterans, and chronic sheltered veterans. Additionally, it calculates the number of unsheltered veterans and chronic unsheltered veterans by subtracting the sheltered counts from the total counts. The task tracker is updated with the status of the calculation process.

In [35]:
task = Task(
    task_nm="Calculate vets that are chronically homeless amd sheltered or unsheltered",
    task_details="",
)
try:
    task_mgr.check_errors()

    sheltered_programs = [
        "Safe Haven",
        "Transitional Housing",
        "Emergency Shelter – Entry Exit",
    ]  # Define the list of sheltered programs

    # Create a new column to store the results

    result_veterans_df["Chronic Vets"] = 0

    result_veterans_df["Sheltered Vets"] = 0

    result_veterans_df["Chronic Sheltered Vets"] = 0

    # Generate the last date of each month for the past 12 months

    for i in range(1, 13):

        last_date_of_month = (now - pd.DateOffset(months=i)).replace(
            day=1
        ) + pd.DateOffset(months=1, days=-1)

        one_year_ago = last_date_of_month - pd.DateOffset(
            years=1
        )  # Calculate the date one year ago

        clients_in_transitional_housing = veterans_df[
            (veterans_df["Enrollments Project Start Date"] <= last_date_of_month)
            & (
                (veterans_df["Enrollments Project Exit Date"] > last_date_of_month)
                | (veterans_df["Enrollments Project Exit Date"].isna())
            )
            & (veterans_df["Programs Project Type Code"] == "Transitional Housing")
        ]  # Filter the data to only include clients who were enrolled in transitional housing at the end of the month

        BNL_eligible_df = veterans_df[
            (veterans_df["Enrollments Project Start Date"] <= last_date_of_month)
            & (
                (veterans_df["Enrollments Project Exit Date"] > last_date_of_month)
                | (veterans_df["Enrollments Project Exit Date"].isna())
            )
            & (veterans_df["Programs Name"] == "Coordinated Entry Program")
            & (veterans_df["Client Assessments Assessment Date"] >= one_year_ago)
            & (veterans_df["Client Assessments Assessment Date"] <= last_date_of_month)
            & (
                veterans_df["Client Assessment Custom Veteran Discharge Status"].isin(
                    eligible_discharges
                )
            )
        ].copy()  # Filter the data to only include clients who were enrolled in the coordinated entry program at the end of the month, had an assessment in the past year, and have an eligible discharge status

        BNL_eligible_df.loc[:, "Chronic?"] = BNL_eligible_df.apply(
            lambda row: (
                "Yes"
                if (
                    row[
                        "Entry Screen Chronically Homeless at Project Start - Individual"
                    ]
                    == "Yes"
                    or (
                        row["Entry Screen Disabling Condition"] == "Yes"
                        and (
                            pd.to_datetime(row["Enrollments Project Start Date"])
                            - pd.to_datetime(
                                row[
                                    "Entry Screen Approximate Date this Episode of Homelessness Started Date"
                                ]
                            )
                        ).days
                        >= 365
                    )
                )
                and (
                    row["Clients Client ID"]
                    not in clients_in_transitional_housing["Clients Client ID"].values
                )
                else "No"
            ),
            axis=1,
        )  # Determine if the client is chronically homeless by checking if they are chronically homeless at project start or have a disabling condition and have been houseless for more than a year

        chronic_vets = BNL_eligible_df[BNL_eligible_df["Chronic?"] == "Yes"]

        count_chronic = BNL_eligible_df[BNL_eligible_df["Chronic?"] == "Yes"][
            "Clients Client ID"
        ].nunique()  # Count the number of unique clients who are chronically homeless

        sheltered_vets = veterans_df[
            (
                veterans_df["Clients Client ID"].isin(
                    BNL_eligible_df["Clients Client ID"]
                )
            )
            & (veterans_df["Programs Project Type Code"].isin(sheltered_programs))
            & (veterans_df["Enrollments Project Start Date"] <= last_date_of_month)
            & (
                (veterans_df["Enrollments Project Exit Date"] > last_date_of_month)
                | (veterans_df["Enrollments Project Exit Date"].isna())
            )
        ]  # Filter the data to only include clients who were on teh BNL for teh month and were enrolled in a sheltered program at the end of the month

        count_sheltered = sheltered_vets[
            "Clients Client ID"
        ].nunique()  # Count the number of unique clients who are sheltered

        count_chronic_sheltered = veterans_df[
            (
                veterans_df["Clients Client ID"].isin(
                    BNL_eligible_df[BNL_eligible_df["Chronic?"] == "Yes"][
                        "Clients Client ID"
                    ]
                )
            )
            & (veterans_df["Programs Project Type Code"].isin(sheltered_programs))
            & (veterans_df["Enrollments Project Start Date"] <= last_date_of_month)
            & (
                (veterans_df["Enrollments Project Exit Date"] > last_date_of_month)
                | (veterans_df["Enrollments Project Exit Date"].isna())
            )
        ][
            "Clients Client ID"
        ].nunique()  # Count the number of unique clients who are chronically homeless and sheltered

        result_veterans_df.loc[last_date_of_month.strftime("%Y-%m"), "Chronic Vets"] = (
            count_chronic  # Append the results to the DataFrame
        )

        result_veterans_df.loc[
            last_date_of_month.strftime("%Y-%m"), "Sheltered Vets"
        ] = count_sheltered  # Append the results to the DataFrame

        result_veterans_df.loc[
            last_date_of_month.strftime("%Y-%m"), "Chronic Sheltered Vets"
        ] = count_chronic_sheltered  # Append the results to the DataFrame

    # Getting unsheltered and unsheltered chronic through subtraction (row 29 and 41)
    result_veterans_df["Unsheltered Vets"] = (
        result_veterans_df["Vets on BNL with Eligible Discharge"]
        - result_veterans_df["Sheltered Vets"]
    )  # Calculate the number of unsheltered veterans

    result_veterans_df["Chronic Unsheltered Vets"] = (
        result_veterans_df["Chronic Vets"]
        - result_veterans_df["Chronic Sheltered Vets"]
    )  # Calculate the number of chronically homeless unsheltered veterans

    # updating the task tracker
    task["Details"] = (
        "Successfully calculated vets that are chronically homeless amd sheltered or unsheltered"
    )
    task["Success"] = True
    logger.info(
        "Calculated vets that are chronically homeless amd sheltered or unsheltered"
    )

except Exception as e:
    task["Details"] = str(e)
    logger.error(
        f"calculating vets that are chronically homeless amd sheltered or unsheltered failed with the following error :{e}"
    )
    print(e)

task_mgr.add_task(task)

Count of inflow and outflow row 51 and 65

This cell calculates the inflow and outflow of veterans on the By-Name List (BNL). It iterates over the past 12 months, generating the last date of each month and filtering the `veterans_df` DataFrame to include only those veterans who were enrolled in a coordinated entry program as of the last day of the month, those with a VI-SPDAT in the past 365 days from the last date of the month, and those with an eligible discharge status. The inflow is calculated as the count of unique client IDs that appear in the current month's BNL but not in the previous month's BNL. The outflow is calculated as the count of unique client IDs that appear in the previous month's BNL but not in the current month's BNL. The results are stored in the `result_veterans_df` DataFrame under the columns "Total Inflow" and "Total Outflow". The task tracker is updated with the status of the calculation process.

In [36]:
task = Task(
    task_nm="Calculate the inflow and outflow of veterans on BNL",
    task_details="",
)
try:
    task_mgr.check_errors()

    # Create a new column to store the results

    result_veterans_df["Total Outflow"] = 0

    result_veterans_df["Total Inflow"] = 0

    # Generate the last date of each month for the past 12 months

    for i in range(1, 13):

        last_date_of_month = (now - pd.DateOffset(months=i)).replace(
            day=1
        ) + pd.DateOffset(months=1, days=-1)

        previous_month = (now - pd.DateOffset(months=i + 1)).replace(
            day=1
        ) + pd.DateOffset(
            months=1, days=-1
        )  # Calculate the last date of the previous month

        one_year_ago = last_date_of_month - pd.DateOffset(years=1)

        previous_one_year_ago = previous_month - pd.DateOffset(years=1)

        previous_month_bnl = veterans_df[
            (veterans_df["Enrollments Project Start Date"] <= previous_month)
            & (
                (veterans_df["Enrollments Project Exit Date"] > previous_month)
                | (veterans_df["Enrollments Project Exit Date"].isna())
            )
            & (veterans_df["Programs Name"] == "Coordinated Entry Program")
            & (
                veterans_df["Client Assessments Assessment Date"]
                >= previous_one_year_ago
            )
            & (veterans_df["Client Assessments Assessment Date"] <= previous_month)
            & (
                veterans_df["Client Assessment Custom Veteran Discharge Status"].isin(
                    eligible_discharges
                )
            )
        ].copy()  # Filter the data to only include clients who were enrolled in the coordinated entry program at the end of the previous month, had an assessment in the past year and have an eligible discharge status

        current_month_bnl = veterans_df[
            (veterans_df["Enrollments Project Start Date"] <= last_date_of_month)
            & (
                (veterans_df["Enrollments Project Exit Date"] > last_date_of_month)
                | (veterans_df["Enrollments Project Exit Date"].isna())
            )
            & (veterans_df["Programs Name"] == "Coordinated Entry Program")
            & (veterans_df["Client Assessments Assessment Date"] >= one_year_ago)
            & (veterans_df["Client Assessments Assessment Date"] <= last_date_of_month)
            & (
                veterans_df["Client Assessment Custom Veteran Discharge Status"].isin(
                    eligible_discharges
                )
            )
        ].copy()  # Filter the data to only include clients who were enrolled in the coordinated entry program at the end of the month, had an assessment in the past year, and have an eligible discharge status

        outflow_vets_bnl = previous_month_bnl[
            ~previous_month_bnl["Clients Client ID"].isin(
                current_month_bnl["Clients Client ID"]
            )
        ]  # Filter the data to only include clients who were on the BNL in the previous month but not in the current month

        count_outflow_bnl = outflow_vets_bnl[
            "Clients Client ID"
        ].nunique()  # Count the number of unique clients who left the BNL

        inflow_vets_bnl = current_month_bnl[
            ~current_month_bnl["Clients Client ID"].isin(
                previous_month_bnl["Clients Client ID"]
            )
        ]  # Filter the data to only include clients who were on the BNL in the current month but not in the previous month

        count_inflow_bnl = inflow_vets_bnl[
            "Clients Client ID"
        ].nunique()  # Count the number of unique clients who joined the BNL

        result_veterans_df.loc[last_date_of_month.strftime("%Y-%m"), "Total Inflow"] = (
            count_inflow_bnl  # Append the results to the DataFrame
        )

        result_veterans_df.loc[
            last_date_of_month.strftime("%Y-%m"), "Total Outflow"
        ] = count_outflow_bnl  # Append the results to the DataFrame

    # updating the task tracker
    task["Details"] = (
        "Successfully calculated the inflow and outflow of veterans on BNL"
    )
    task["Success"] = True
    logger.info("Calculated the inflow and outflow of veterans on BNL")

except Exception as e:
    task["Details"] = str(e)
    logger.error(
        f"calculating the inflow and outflow of veterans on BNL failed with the following error :{e}"
    )
    print(e)

task_mgr.add_task(task)

We couldn't get the drilldown on inflow and outflow working. If you want to add it yourself, here's how:

1. Update `previous_month_bnl` and `current_month_bnl` to have a month aware chronic column. Refer to the above section for details. Consider putting this in a function to reduce repeated code.
2. Recalculate `outflow_vets_bnl` and `inflow_vets_bnl`.
3. For `inflow_vets_bnl`:
    - Count how many have a VI-SPDAT assessment done in the current month and how many are chronic. Remove them.
    - For the remaining, count how many have been in PSH or RRH programs using a helper DataFrame similar to the one for transitional housing. Count them and any who are chronic. Remove them.
    - For the remaining, see if they've been in other programs using a helper DataFrame. Count chronic and the remaining.

4. For `outflow_vets_bnl`:
    - Check if they are in US Vets, Catholic Charities, or VASH and if they have a referral. Remove them.
    - For the remaining, check their most recent VI-SPDAT. If it's over a year ago, count them and chronic. Remove them.
    - Count any remaining for the last row.

This cell uploads the aggregated results stored in the `result_veterans_df` DataFrame to the `partners_in_care_veterans_summary` table in the database. It resets the index of the DataFrame, adds update timestamps in HST, CST, and UTC time zones, and then inserts the data into the database using the `GsAutomationReliefAssistanceConnection` class. The task tracker is updated with the status of the database insertion process.


In [37]:
task = Task(
    task_nm="Insert data into the database",
    task_details="",
)
try:
    task_mgr.check_errors()

    result_veterans_df.reset_index(inplace=True)

    result_veterans_df["Update Time HST"] = now_hst_str

    result_veterans_df["Update Time CST"] = now_str_formatted

    result_veterans_df["Update Time UTC"] = now_utc_str

    conn = PicHorneDashesConnection()

    conn.insert_with_progress(
        df=result_veterans_df,
        schema="dbo",
        tbl_nm="partners_in_care_veterans_summary",
        dtypes={},
    )

    # updating the task tracker
    task["Details"] = "Successfully inserted data into the database"
    task["Success"] = True
    logger.info("Inserted data into the database")

except Exception as e:
    task["Details"] = str(e)
    logger.error(
        f"inserting data into the database failed with the following error :{e}"
    )
    print(e)

task_mgr.add_task(task)

[Attempt 1] Database connection failed. Retrying in 60 seconds...


100%|██████████| 12/12 [00:03<00:00,  3.33it/s]


This cell sends out an alert email if any of the above cells failed

In [38]:
if task_mgr._process_success != "Success":
    subject, body = task_mgr.generate_task_email(today_str)
    email_client = DsEmail()
    result = email_client.send_task_alert_email(subject, body, "partners_in_care")

Close out the logger

In [39]:
logger_manager.cleanup()